In [22]:
import pandas as pd
import numpy as np

from sklearn.metrics import mean_absolute_percentage_error

from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.holtwinters import ExponentialSmoothing
!pip install openpyxl


[notice] A new release of pip is available: 24.0 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [23]:
# -------------------------------------------------------------
# STEP 1: LOAD DATA
# -------------------------------------------------------------

file_path = "interview question.xlsx"

raw = pd.read_excel(file_path, header=None)


In [24]:
# Header row is row 6 in the spreadsheet
header = raw.iloc[6, 1:].tolist()

data = raw.iloc[7:, 1:].copy()
data.columns = header

print("Shape:", data.shape)
print(data.head())


Shape: (10, 37)
       Product 2022-01 2022-02 2022-03 2022-04 2022-05 2022-06 2022-07  \
7    Product_1      51      51      54      58      54      55      62   
8   Product_10      48      49      53      62      69      70      72   
9    Product_2      81      80      91     101     101      91      79   
10   Product_3      30      46      30      36      25      24      35   
11   Product_4      64      59      31      60      61      97      57   

   2022-08 2022-09  ... 2024-03 2024-04 2024-05 2024-06 2024-07 2024-08  \
7       61      58  ...      78      84      82      84      84      93   
8       69      71  ...     139     137     144     146     159     157   
9       68      55  ...      92      94     101      97      80      75   
10      36      44  ...      59      51      38      43      53      60   
11      65      59  ...      46      83      48      55      72      42   

   2024-09 2024-10 2024-11 2024-12  
7       88      86      93      88  
8      155    

In [25]:
# -------------------------------------------------------------
# STEP 2: CONVERT TO LONG FORMAT
# -------------------------------------------------------------

long_df = data.melt(
    id_vars="Product",
    var_name="Month",
    value_name="Sales"
)

long_df["Month"] = pd.to_datetime(long_df["Month"])
long_df["Sales"] = pd.to_numeric(long_df["Sales"])

long_df = long_df.sort_values(["Product", "Month"])

print("\nLong format sample:\n")
print(long_df.head())


Long format sample:

      Product      Month  Sales
0   Product_1 2022-01-01     51
10  Product_1 2022-02-01     51
20  Product_1 2022-03-01     54
30  Product_1 2022-04-01     58
40  Product_1 2022-05-01     54


In [26]:
# -------------------------------------------------------------
# STEP 3: QUICK DATA QUALITY CHECKS
# -------------------------------------------------------------

print("\nMissing values:\n")
print(long_df.isna().sum())

print("\nZero counts per product:\n")
print(
    long_df.groupby("Product")["Sales"]
    .apply(lambda x: (x == 0).sum())
    .sort_values(ascending=False)
)



Missing values:

Product    0
Month      0
Sales      0
dtype: int64

Zero counts per product:

Product
Product_9     28
Product_5     20
Product_1      0
Product_10     0
Product_2      0
Product_3      0
Product_4      0
Product_6      0
Product_7      0
Product_8      0
Name: Sales, dtype: int64


In [27]:
# -------------------------------------------------------------
# STEP 4: FEATURE ENGINEERING
# -------------------------------------------------------------
#
# Interview answer:
#
# Lag Features:
#   Sales(t-1), Sales(t-2), Sales(t-3)
#
# Rolling Mean:
#   3-month rolling average
#
# Rolling Std:
#   Volatility feature
#
# Trend:
#   Time index
#
# -------------------------------------------------------------

long_df["lag_1"] = long_df.groupby("Product")["Sales"].shift(1)
long_df["lag_2"] = long_df.groupby("Product")["Sales"].shift(2)
long_df["lag_3"] = long_df.groupby("Product")["Sales"].shift(3)

long_df["rolling_mean_3"] = (
    long_df.groupby("Product")["Sales"]
    .transform(lambda x: x.rolling(3).mean())
)

long_df["rolling_std_3"] = (
    long_df.groupby("Product")["Sales"]
    .transform(lambda x: x.rolling(3).std())
)

long_df["time_index"] = (
    long_df.groupby("Product")
    .cumcount()
)

print("\nFeature engineering complete")


Feature engineering complete


In [28]:
# -------------------------------------------------------------
# STEP 5: MODEL SELECTION LOGIC
# -------------------------------------------------------------
#
# Product observations:
#
# Product_1:
#     steady upward trend
#
# Product_10:
#     strong upward trend
#
# Product_2:
#     seasonality-like behavior
#
# Product_5:
#     many zeros
#
# Product_9:
#     intermittent demand
#
# Interview answer:
#
# - Trend products -> ARIMA / Holt-Winters
# - Stable products -> Exponential Smoothing
# - Sparse products -> Moving Average baseline
#
# -------------------------------------------------------------

def forecast_product(series, horizon=6):

    series = series.astype(float)

    train = series[:-6]
    test = series[-6:]

    results = {}

    # ---------------------------------------------------------
    # MODEL 1: Naive Forecast
    # ---------------------------------------------------------

    naive_pred = np.repeat(train.iloc[-1], len(test))

    naive_mape = mean_absolute_percentage_error(
        test,
        naive_pred
    )

    results["Naive"] = naive_mape

    # ---------------------------------------------------------
    # MODEL 2: Exponential Smoothing
    # ---------------------------------------------------------

    try:
        hw = ExponentialSmoothing(
            train,
            trend="add",
            seasonal=None
        ).fit()

        hw_pred = hw.forecast(len(test))

        hw_mape = mean_absolute_percentage_error(
            test,
            hw_pred
        )

        results["Holt"] = hw_mape

    except:
        pass

    # ---------------------------------------------------------
    # MODEL 3: ARIMA
    # ---------------------------------------------------------

    try:

        model = ARIMA(
            train,
            order=(1,1,1)
        ).fit()

        pred = model.forecast(len(test))

        arima_mape = mean_absolute_percentage_error(
            test,
            pred
        )

        results["ARIMA"] = arima_mape

    except:
        pass

    best_model = min(results, key=results.get)

    return results, best_model



In [29]:
# -------------------------------------------------------------
# STEP 6: MODEL COMPARISON
# -------------------------------------------------------------

summary = []

for product in data["Product"]:

    series = (
        long_df[long_df["Product"] == product]
        .sort_values("Month")["Sales"]
        .reset_index(drop=True)
    )

    scores, best_model = forecast_product(series)

    summary.append({
        "Product": product,
        "Best_Model": best_model,
        **scores
    })

summary_df = pd.DataFrame(summary)

print("\nModel comparison:\n")
print(summary_df)

C:\Users\bangl\AppData\Local\Programs\Python\Python312\Lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
C:\Users\bangl\AppData\Local\Programs\Python\Python312\Lib\site-packages\statsmodels\tsa\statespace\sarimax.py:966: UserWarning: Non-stationary starting autoregressive parameters found. Using zeros as starting parameters.
  warn('Non-stationary starting autoregressive parameters'
C:\Users\bangl\AppData\Local\Programs\Python\Python312\Lib\site-packages\statsmodels\tsa\statespace\sarimax.py:978: UserWarning: Non-invertible starting MA parameters found. Using zeros as starting parameters.
  warn('Non-invertible starting MA parameters found.'
C:\Users\bangl\AppData\Local\Programs\Python\Python312\Lib\site-packages\statsmodels\tsa\statespace\sarimax.py:978: UserWarning: Non-invertible starting MA parameters found. Using zeros as starting pa


Model comparison:

      Product Best_Model         Naive          Holt         ARIMA
0   Product_1       Holt  5.128555e-02  3.134916e-02  5.766592e-02
1  Product_10      Naive  4.478897e-02  6.552538e-02  6.295079e-02
2   Product_2       Holt  4.137379e-01  3.556578e-01  3.684571e-01
3   Product_3       Holt  2.384835e-01  9.729821e-02  2.339050e-01
4   Product_4      ARIMA  2.733135e-01  2.722196e-01  2.647553e-01
5   Product_5      Naive  6.666667e-01  2.720710e+16  2.921349e+16
6   Product_6       Holt  1.779043e-01  1.493375e-01  1.823771e-01
7   Product_7      ARIMA  6.892972e-02  7.491745e-02  6.537214e-02
8   Product_8       Holt  1.194444e+00  7.765635e-01  1.107590e+00
9   Product_9      ARIMA  1.711368e+17  6.162193e+16  5.769904e+16


In [20]:
# -------------------------------------------------------------
# STEP 7: FINAL FORECAST
# -------------------------------------------------------------

future_forecasts = []

for product in data["Product"]:

    series = (
        long_df[long_df["Product"] == product]
        .sort_values("Month")["Sales"]
        .reset_index(drop=True)
        .astype(float)
    )

    try:

        model = ARIMA(
            series,
            order=(1,1,1)
        ).fit()

        forecast = model.forecast(6)

        for i, value in enumerate(forecast, start=1):

            future_forecasts.append({
                "Product": product,
                "Month_Ahead": i,
                "Forecast": round(float(value), 2)
            })

    except:

        last_val = series.iloc[-1]

        for i in range(1, 7):

            future_forecasts.append({
                "Product": product,
                "Month_Ahead": i,
                "Forecast": round(float(last_val), 2)
            })

forecast_df = pd.DataFrame(future_forecasts)

print("\nFuture Forecasts:\n")
print(forecast_df.head(20))

forecast_df.to_csv(
    "forecast_output.csv",
    index=False
)

print("\nSaved forecast_output.csv")


C:\Users\bangl\AppData\Local\Programs\Python\Python312\Lib\site-packages\statsmodels\tsa\statespace\sarimax.py:966: UserWarning: Non-stationary starting autoregressive parameters found. Using zeros as starting parameters.
  warn('Non-stationary starting autoregressive parameters'



Future Forecasts:

       Product  Month_Ahead  Forecast
0    Product_1            1     89.50
1    Product_1            2     89.21
2    Product_1            3     89.26
3    Product_1            4     89.25
4    Product_1            5     89.25
5    Product_1            6     89.25
6   Product_10            1    143.62
7   Product_10            2    142.41
8   Product_10            3    141.37
9   Product_10            4    140.47
10  Product_10            5    139.69
11  Product_10            6    139.01
12   Product_2            1     84.61
13   Product_2            2     86.85
14   Product_2            3     87.61
15   Product_2            4     87.87
16   Product_2            5     87.96
17   Product_2            6     87.99
18   Product_3            1     50.90
19   Product_3            2     51.36

Saved forecast_output.csv
